<img src="images/scilife_logo.png" width="400">
<img src="images/essence_logo.png" width="300">

# SciLifeLab Workshop - Hands-on Section: LangGraph "Hello World"



Welcome to this hands‑on lab session on building AI Agents with **LangGraph**! 

LangGraph is a low‑level orchestration framework for constructing stateful AI workflows using graphs. LangGraph provides several key benefits for agentic AI applications, including customizability, durable execution, support for human‑in‑the‑loop workflows, comprehensive memory (both short‑ and long‑term), built‑in debugging, and production‑ready deployment features.

## Learning Objectives
By the end of this workshop, you will:
- Understand core concepts of LangGraph (tools, nodes, edges, state, and memory).
- Build a ReAct‑style agent using an LLM and custom tools.
- Implement agent memory to maintain conversational context.
- Compare custom agents with prebuilt agents (`create_agent`).
- *(Optional)* Explore advanced agent architectures, including the CodeAct agent, human‑in‑the‑loop approval and multi‑agent systems.
- *(Optional)* Extend prebuilt agents with structured output, middleware and long‑term memory.

## Workshop Outline
- **Part 1**: Setup & imports
- **Part 2**: Understanding and creating tools
- **Part 3**: Defining the state
- **Part 4**: Building the agent graph
- **Part 5**: Testing the agent
- **Part 6**: Adding short‑term memory
- **Part 7**: Exploring prebuilt agents (`create_agent`)
- **Part 8 (optional)**: Extension exercises: 

    + CodeAct agent
    + Multi‑agent supervisor architectrure, 
    + Structured output
    + Middleware
    + Long‑term memory

<img src="images/outline.png" width="800">

## Instructions for Participants

**Throughout this lab, look for `TODO` comments and `...` placeholders in the code cells. These indicate where you need to add your own code.** 

Each exercise builds on the previous one, so complete them in order!

Your task is to:
1. Replace `...` with appropriate code
2. Follow the instructions in `TODO` comments 
3. If you are stuck, feel free to find information internet, the answer notebook, or reach out to the instructor

---

## Part 1: Setup

### Exercise 1.1: Import dependencies

In [ ]:
import time
from dotenv import load_dotenv
from langchain.tools import tool

# Load the environment variables
load_dotenv(override=True)

### Exercise 1.2: Configure the LLM

In this workshop we use the **SciLifeLab openLLM service** by default (model `Qwen/Qwen3.8-27B`).

The function `get_llm()` below helps you call the LLM that will be used for agent later. It will tries the pilot service first; if it cannot be reached during the session (or no `PILOT_API_KEY` is set), it automatically falls back to OpenAI. 

In [ ]:
import os
from langchain_openai import ChatOpenAI

# LLM configuration: SciLifeLab Open LLM service
OPENLLM_BASE_URL = "https://openllm.scilifelab.se/api"
OPENLLM_MODEL = "Qwen/Qwen3.8-27B"  # or gemma-27b, Qwen3-235B-A22B

def get_llm(model= OPENLLM_MODEL, base_url = OPENLLM_BASE_URL, temperature: float = 0, **kwargs):
    """Return a chat model, preferring the SciLifeLab pilot service.
    """
    api_key = os.getenv('OPENLLM_API_KEY')

    llm = ChatOpenAI(
            model=model,
            base_url=base_url,
            api_key=api_key,
            temperature=temperature,
            extra_body = {"chat_template_kwargs": {"enable_thinking": False}},
            **kwargs,
        )
    try:
      llm.invoke("ping")  # lightweight connectivity check
      print(f"✓ Using SciLifeLab pilot LLM ({OPENLLM_MODEL})")
      return llm
    except Exception as e:
      print(f"⚠ Pilot LLM unavailable ({type(e).__name__}: {e}).")

# Test connection to LLM
llm = get_llm()

---

## Part 2: Understanding and Creating Tools

In LangGraph, **tools** are Python functions that extend your agent's capabilities beyond text generation. They can call external APIs or perform computations, and are annotated with the `@tool` decorator from LangChain.

When designing tools for agents, there are two important considerations:
- **Input - Output**: It is similar to when designing a Python function. While being used, the input is generated by LLM (rather than human typing). Output will be fed back to agent's context.

- **Description**: The `@tool` decorator requires docstrings from a Python function and will use it as the tool's description. These descriptions will feed the LLM system prompts, making it aware of these existing tools and their schema. With `@tool(parse_docstring=True)`, each entry of the Google‑style `Args:` section also becomes the description of that argument.

<img src="images/exercise_2_intro.png" width="800">

### Exercise 2.1: Import the tools

In this exercise, we have already prepared tools in areas of drug discovery/life sciences for you at `utils/tools/`. 

Your task is to:
- Go to the python scripts at `utils/tools/`
- Skim them to find the tool names (you do not need to understand each script in full, but you are welcome to read them)
- Import all the tools to this notebook 9only the tools (do not import all the functions in the scripts)

In [ ]:
#TODO: Import tools from utils/tools/*.py to this notebook
from utils.tools.chemistry import resolve_smiles, ...
from utils.tools.biology import ...
from utils.tools.pkpd import ...
from utils.tools.literature import ...

### Exercise 2.2: Inspect what does the LLM actually see?

The `@tool` decorator turns each function into a `StructuredTool` object with a **name**, a **description** and a **JSON schema** for its arguments. 

This information will be sent to the LLM, so that it will be aware of the tools existence and how to call them. 

In [ ]:
import json

# The json schema that is actually sent to the LLM for resolve_smiles tool
print(json.dumps(resolve_smiles.tool_call_schema.model_json_schema(), indent=2))

In [ ]:
#TODO: inspect any tool you want (try 1-2 tools then you should move on to the next task)
print(json.dumps(....tool_call_schema.model_json_schema(), indent=2))

### Exercise 2.3: Create a Tools List

In [ ]:
#TODO: Collect all of your tool functions into a single list called `TOOLS`. 
TOOLS = [resolve_smiles, ...]

---

## Part 3: Understanding LangGraph State

The LLM itself remembers nothing between calls, it only sees what you send it and produces an output from that input alone. There is no context management inside the model. In LangChain and LangGraph, the state holds a running list of messages that grows over the conversation and is handed to the model at every step.

When creating a state schema, you use `TypedDict` to describe the keys and `Annotated` with updating functions to specify how values should be updated.

The updating function `add_messages` appends new messages into the Agent's State. It also converts plain dicts such as `{'role': 'user', 'content': '...'}` into one of the LangChain **message objects** below:

| Message class | Who produced it | Key fields |
|---|---|---|
| `SystemMessage` | the user| `content` |
| `HumanMessage` | the user | `content` |
| `AIMessage` | the LLM | `content`, `tool_calls` |
| `ToolMessage` | a tool | `content`, `tool_call_id` (links the result to the call that requested it) |

`AnyMessage` is the type that covers all of them.

### Exercise 3.1: Define Chat State

Define a `ChatState` class (subclassing `TypedDict`) with a single key `messages`. Use the `add_messages` updating function so that new messages are appended to the state.

<img src="images/ChatState.png" width="800">

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    """The state schema for our LangGraph. It contains only a list of messages.

    Messages are appended via the `add_messages` to preserve the full conversation history."""
    messages: Annotated[list[AnyMessage], add_messages]

### Exercise 3.2: See the updating function in action

After every node, LangGraph merges the node's output into the State using the updating function. You can call `add_messages` yourself to see what happens.

In [ ]:
from langchain_core.messages import AIMessage

# Initial state
existing = [{'role': 'user', 'content': 'Is Erlotinib orally drug-like?'}] 

# Add-up messages 
turn_1 = [AIMessage(content='Let me compute its properties first.')] 
turn_2 = [AIMessage(content='The properties looks good with no Lipinski rule violation.')]       
turn_3 = [AIMessage(content='Yes, Erlotinib is a drug-like molecule.')]

# Print out the evolving process of the agent state
for i, add_up_msg in enumerate([[],turn_1, turn_2, turn_3]):
    print(' ')
    print(f'Turn {i}')
    existing = add_messages(existing, add_up_msg)

    for m in existing:
        print(m)

---

## Part 4: Building the Agent Graph

This is the graph we are going to create: 

<img src="images/react_agent.png" width="300">

### Exercise 4.1: Initialize Components

1. Initialise a `StateGraph` instance using your `ChatState`.
2. Initialise a chat model with the `get_llm()` helper from Part 1.
3. Bind your tools to the LLM using `bind_tools()`.
4. Create a `ToolNode` from your tools list.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

# Create graph builder using ChatState
graph_builder = StateGraph(ChatState)

#TODO: Initialise the chat model using the get_llm() function from Part 1
llm = get_llm()

#TODO: Bind the tools list to the LLM so that it knows their schemas and how to construct tool calls in JSON
llm_with_tools = llm.bind_tools(..., parallel_tool_calls=False)

#TODO: Create a ToolNode using the tools list
tool_node = ToolNode(tools=...)

### Exercise 4.2: Define the Chatbot Node

Define a chatbot function that takes the `state` as input and returns a dictionary with a single key `messages`.

When invoke the LLM, we provide it with all the `messages` in `state`.

<img src="images/chatbot.png" width="800">
<img src="images/chatbot2.png" width="800">

In [ ]:
def chatbot(state: ChatState) -> dict:
    """The main chatbot node. It invokes the LLM with the current messages."""
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

### Exercise 4.3: Define the Routing Function !!! IMPORTANT

Create a function `route_tools` that decides whether to call tools or to terminate the process.

- Step 1: Extract the last message from `state`.
- Step 2: If the last message contains tool calls (`tool_calls` attribute), return `'tools'`; otherwise return `END`.

In [ ]:
#TODO: complete the output of route function
def route_tools(state: ChatState):
    """Send the conversation to the tool node if the LLM asked for a tool, otherwise finish."""
    
    # Get the messages list in the agent state
    messages = state.get('messages', [])
    
    # Get the last AI_message
    ai_message = messages[-1] if messages else None

    # If the last message contains tool calls (`tool_calls` attribute), return `'tools'`; otherwise return `END`.
    if ...:
        return ...
    return ...

### Exercise 4.4: Build the Complete Graph

Assemble the agent graph by adding nodes and edges, then compile the graph into a runnable agent:
- Use `add_node()` for your chatbot and tool nodes
- Use `add_edge()` to connect nodes.
- Use `add_conditional_edges()` for routing logic

<img src="images/graph_create_1.png" width="800">

<img src="images/graph_create_2.png" width="800">


This is the graph we are going to create: 

<img src="images/react_agent.png" width="300">

In [ ]:
graph_builder = StateGraph(ChatState)

# TODO: Add the chatbot node to the graph
graph_builder.add_node('...', ...)

# TODO: Add the tool node to the graph
graph_builder.add_node('...', ...)

# TODO: Add conditional edges from chatbot using the routing function
graph_builder.add_conditional_edges('...', route_tools, {'...': '...', '...': ...})

# TODO: Add edge from tools back to chatbot
graph_builder.add_edge('...', '...')

# TODO: Add edge from START to chatbot
graph_builder.add_edge(..., '...')

# Compile the graph into an agent 
agent = graph_builder.compile()

### Exercise 4.5: Visualize Your Agent

Use the graph's `draw_mermaid_png()` method to visualize the structure of your agent. This step is helps you understand how nodes and edges connect.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print('Graph visualization not available in this environment.')

---

## Part 5: Testing Your Agent

### Exercise 5.1: Interact with Basic Chat Loop

Try out this chat loop with the prompts below, grouped by domain. You are welcome to try any promtps you want. 

Since we haven't talked about the system prompt yet, remember to add this before your own prompt: **"You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data."**

#### Chemistry

- **Prompt 1**: ” You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Here is the compound ID: Erlotinib. Is it orally drug-like?”

- **Prompt 2**: " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Which is more lipophilic: Imatinib or Dasatinib?”

- **Prompt 3**: " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Check betulinic acid. Are there any structural red flags for oral bioavailability?”

- **Prompt 4**: " You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. What are the common physicochemical characteristics of successful JAK inhibitors?"

- **Prompt 5**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Rank Gefitinib, Lapatinib, Afatinib, Osimertinib and Imatinib by structural similarity to Erlotinib. Which ones share Erlotinib's scaffold, and does structural similarity match whether they act on the same target?"

#### Biology

- **Prompt 6**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. What are the molecular targets and mechanism of action of Imatinib? Describe the biological function and subcellular location of its main target."

- **Prompt 7**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Which proteins interact most strongly with EGFR? Search the literature to check whether any of these partners are linked to resistance to EGFR inhibitors such as Erlotinib."

- **Prompt 8**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Do Erlotinib and Gefitinib act on the same target? Compare their mechanisms of action, approved indications and oral drug-likeness."

- **Prompt 9**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Compare native GLP-1(7-37) (HAEGTFTSDVSSYLEGQAAKEFIAWLVKGRG) with the drug exenatide (HGEGTFTSDLSKQMEEEAVRLFIEWLKNGGPSSGAPPPS): length, molecular weight, isoelectric point, hydrophobicity and predicted stability. Then use the literature to explain why exenatide lasts much longer in the body than native GLP-1. Does the sequence-based stability prediction agree?"

#### PK/PD simulation

- **Prompt 10**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. Imatinib is given orally at 400 mg once daily. A population PK study (Widmer et al., Br J Clin Pharmacol 2006) reports an apparent oral clearance CL/F = 14.3 L/h and volume V/F = 347 L; assume ka = 0.6 1/h. Simulate two weeks of dosing. Does the steady-state trough stay above 1 mg/L (1000 ng/mL), the threshold associated with major molecular response in CML (Picard et al., Blood 2007)? If not, which once-daily dose, in steps of 100 mg, would reach it?"

- **Prompt 11**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. What is the molecular target of Erlotinib? Then simulate 150 mg once daily for 3 weeks using its population PK parameters CL/F = 3.95 L/h, V/F = 233 L and ka = 0.95 1/h (Lu et al., Clin Pharmacol Ther 2006). How long does it take to reach steady state, and what are the steady-state Cmax and trough?"

- **Prompt 12**: "You are an expert drug discovery researcher. Use your available tools to answer the user’s question as accurately as possible. Never fabricate or invent data. A hypothetical oral kinase inhibitor has CL/F = 10 L/h, V/F = 100 L and ka = 1 1/h. Its target inhibition follows an Emax model with Emax = 100 %, EC50 = 0.5 mg/L and Hill coefficient 1. Compare 300 mg once daily with 150 mg twice daily over one week: which regimen keeps target inhibition above 50 % for a larger part of the dosing interval at steady state?"

*Notes on the PK/PD prompts:* the agent has no database of PK parameters, so the prompts provide them. The simulation tools also save a plot in `outputs/pkpd/` and return its path: open it from the file browser on the left. 

*Challenge:* remove the numbers from Prompt 10-12 and ask the agent to find them first. Is there any change in the results?

In [ ]:
print('Welcome to the LangGraph ReAct demo! Ask me a question.')
print('I can look up and compare compounds, analyse proteins and drug targets, simulate PK/PD, and search PubMed.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break
    
    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }



    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    # Stream the answer from agent
    for event in agent.stream(state):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

---

## Part 6: Adding Short‑term Memory

To demonstrate the lack of memory in the agent, please **go back to Part 5**, then perform these two prompts: 

- **Prompt 1:** Here is the compound ID: Erlotinib. Is it orally drug-like?

- **Prompt 2:** Which is more lipophilic: Imatinib or the previous drug?

You will see when running **prompt 2**, the agent get confuse since it doesn't know the previous drug is Erlotinib. This is because the GraphState is only maintained from the START node to the END node. After termination, all contexts from the previous run are lost. Therefore, we need a component called Agent Memory to maintain context across runs.

Memory allows your agent to remember previous parts of the conversation. LangGraph uses "checkpointers" to maintain state across interactions.

### Exercise 6.1: Add Memory to Your Agent

Rebuild your agent with memory by creating a new `StateGraph` that uses the same `ChatState`. 

Initialize `checkpointer` with `InMemorySaver` to persist state across turns.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

graph_builder = StateGraph(ChatState)

llm = get_llm()
llm_with_tools = llm.bind_tools(TOOLS, parallel_tool_calls=False)

def chatbot(state: ChatState) -> dict:
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

tool_node = ToolNode(tools=TOOLS)

graph_builder.add_node('chatbot', chatbot)
graph_builder.add_node('tools', tool_node)
graph_builder.add_conditional_edges('chatbot', route_tools, {'tools': 'tools', END: END})
graph_builder.add_edge('tools', 'chatbot')
graph_builder.add_edge(START, 'chatbot')

# Define checkpointer
checkpointer = InMemorySaver()
# Use checkpointer in agent compiling
agent = graph_builder.compile(checkpointer=checkpointer)

### Exercise 6.2: Create a Memory‑Enabled Chat Loop

Write a chat loop similar to Part 5, but supply a `config` dictionary to `agent.stream()`. In the config, there are two important parameters, including `thread_id` and `recursion_limit`.

Every run that uses the same `thread_id` continues the same conversation; a new `thread_id` starts a fresh one.

<img src="images/short_term_mem.png" width="800">

In [ ]:
# TODO: Create config for memory
config = {
    'configurable': {
        'thread_id': '....',
    },
    'recursion_limit': ...
}

Complete the code cell below and re-run these two prompts: 
- **Prompt 1:** Here is the compound ID: Erlotinib. Is it orally drug-like?

- **Prompt 2:** Which is more lipophilic: Imatinib or the previous drug?

In [ ]:
print('Welcome to the LangGraph ReAct demo with memory! Ask me a question.')
print('I can look up and compare compounds, analyse proteins and drug targets, simulate PK/PD, and search PubMed.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break

    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }


    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    
    # TODO: Supply config for the agent stream process
    for event in agent.stream(state, config=...):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

---

## Part 7: Prebuilt Agents

LangGraph provides prebuilt agents that implement common architectures such as the ReAct pattern. These agents are quick to set up and let you focus on your tools and prompts rather than on graph wiring.

### Exercise 7.1: Create a Prebuilt Agent

Use `create_agent` to instantiate a prebuilt ReAct agent. Provide your LLM, tools list, and a custom system prompt that instructs the agent how to behave.

#### References: [LangGraph Prebuilt Agent](https://docs.langchain.com/oss/python/langchain/agents)

In [ ]:
# Create config for memory
config = {
    'configurable': {
        'thread_id': 'example_conversation',
    },
    'recursion_limit': 25
}

In [ ]:
from langchain.agents import create_agent

llm = get_llm()
checkpointer = InMemorySaver()

#TODO: fill in the llm, tools list, and system prompt 
agent = create_agent(
    model = llm,
    tools = ...,
    system_prompt="...",
    checkpointer = checkpointer
)

In [ ]:
# create_agent returns a compiled LangGraph graph: compare it with the one you built in Part 4
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    print('Graph visualization not available in this environment.')

In [ ]:
print('Welcome to the prebuilt ReAct agent with memory! Ask me a question.')
print('I can look up and compare compounds, analyse proteins and drug targets, simulate PK/PD, and search PubMed.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break

    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }

    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    
    # Supply config for the agent stream process
    for event in agent.stream(state, config=config):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

---

## Part 8: Extension Exercises (optional)


### Exercise 8.1: CodeAct agent

So far, our **ReAct** agent has picked from a fixed list of tools (the TOOLS list you define at the Exercise 2). This approach has two limitations:

1. **No tool-fixing mechanism:** The predefined tools aren't always a perfect fit for the current dataset, and a ReAct agent has no way to modify its tools or manipulate data flexibly when they fall short.
2. **Fixed tool space:** Once the tools are defined, that's all the agent has. Any task beyond the tool set is simply out of reach.

A **CodeAct** agent addresses both problems: instead of calling a predefined function, it writes code and runs it. As a result, rather than relying on a long list of specialized tools, a minimal CodeAct design needs only two:


1. `python_executor`: executes Python code.
2. `read_skill`: loads the full skill Markdown files into the LLM's context, so the agent knows what's available and some basic isntruction

The possible concern here is that giving an LLM free rein to generate code tends to produce unreliable output that doesn't follow any consistent standard or domain expertise. To keep the agent from reinventing the wheel each time, we give it a small library of **skills**, which includes short Markdown recipes it can read on demand and adapt to the task at hand.


<img src="images/code_act.png" width="800">

Skills follow the standard Claude-skill layout:

```
utils/codeact_skills/skills/
├── chemistry/             ← skill name as the folder name
│   └── SKILL.md           ← starts with YAML frontmatter: name + description
├── biology/
│   └── SKILL.md
├── pkpd/
│   └── SKILL.md
├── literature/
│   └── SKILL.md
└── data_visualization/
    ├── SKILL.md
    └── scripts/
```

For each skill, only `name` and `description` are injected to the agent's system prompt, so the agent know what exists. During the run time, agent dynamically decides which skills to load to the context.

We recommendyou to have a look at these skills at `utils/codeact_skills/`

In [ ]:
from pathlib import Path

# Helper function to parse SKILLs.md
SKILLS_DIR = Path("utils/codeact_skills/skills")

def _parse_frontmatter(text: str) -> dict:
    """Extract YAML frontmatter (between '---' fences) as a dict."""
    if not text.startswith("---"):
        return {}
    end = text.find("\n---", 3)
    if end == -1:
        return {}
    return dict(
        (k.strip(), v.strip())
        for line in text[3:end].strip().splitlines()
        if ":" in line
        for k, _, v in [line.partition(":")]
    )


def load_skill_index() -> list[dict]:
    """Return [{name, description, path}, ...] for every skill folder."""
    return [
        {
            "name": meta.get("name", skill_file.parent.name),
            "description": meta.get("description", ""),
            "path": skill_file,
        }
        for skill_file in sorted(SKILLS_DIR.glob("*/SKILL.md"))
        for meta in [_parse_frontmatter(skill_file.read_text())]
    ]

In [ ]:
# Parse SKILL.md and establish skill list (SKILL_CATALOG)
skill_index = load_skill_index()
SKILL_CATALOG = "\n".join(
    f"- {s['name']}: {s['description']}" for s in skill_index
)
print("Available skills:\n" + SKILL_CATALOG)

# Define tool list for CodeAct Agent:
from utils.codeact_skills.python_executor import python_executor
from utils.codeact_skills.read_files import read_skill

CODEACT_TOOLS = [read_skill, python_executor]

Below is the example system prompt for CodeAct Agent. You could try with it first, then feel free to change the system prompt and observe how the agent performance/behavior change. 

In [ ]:
codeact_system_prompt = f"""You are a drug-discovery research assistant operating as a CodeAct agent: you reason in natural language and act by writing Python code that runs in a stateful executor. 
Your job is to find the smartest, fastest path to a correct, evidence-backed answer.

## Available skills
Each skill is a SKILL.md file containing domain knowledge, API conventions, and worked example code. Load a skill by calling `read_skill('<name>')`.
{SKILL_CATALOG}

Skills are your reliability backbone: when one applies, it encodes the right libraries, the right conventions, and patterns. 
Prefer skill-grounded code over improvisation whenever a skill fits. 
But skills are a *floor*, not a *ceiling*. You are a competent Python programmer and chemist, and the catalog will not cover every task.

## Startup triage
Before writing analysis code, take one short pass to plan:
1. **Identify what the task actually needs**: what data, what computation, what domain (cheminformatics, ML, statistics, file parsing, plotting, etc.).
2. **Match against the catalog.** For each relevant area, load the matching skill via `read_skill(...)` *before* writing code in that area.
3. **Decide your mode:**
   - **Skill-grounded mode** — one or more skills match. Adapt their example code; do not reinvent what a skill already specifies.
   - **Out-of-catalog mode** — no skill applies, or only part of the task is covered. Proceed with first-principles Python using standard libraries (rdkit, pandas, numpy, scipy, sklearn, matplotlib, etc. as available). When you do this, **briefly note in your final answer that this part of the work was outside the skill set** so the user knows the reliability profile.
   - **Hybrid** — load the skills that apply, write original code for the rest. State which is which when it matters.

This triage is not a wall. If midway through you realize a different skill applies, load it then. Skills are cheap.

## The execute → reason loop
Work in small, verifiable steps.

1. **Execute.** Write a short Python snippet and run it via `python_executor`. Each snippet targets ONE sub-goal (load a dataset, inspect its shape, compute one metric, generate one plot). Variables persist across executions — build state incrementally.
2. **Reason.** After each execution, examine the output before doing anything else. State in plain language: what you observed, whether it matches your expectation, what it implies, and what the next step is. If the output is an error or surprise, diagnose it before proceeding — never blindly retry or paper over it.
3. **Iterate.** Use that reasoning to decide the next execution. Repeat until you have enough evidence to answer.

### Rules for the loop
- **Inspect before you compute.** First execution on any new data (uploaded file, API response, query result) examines structure — shape, dtypes, head, schema — before any analysis runs against it.
- **One thing at a time.** Don't chain ten operations into one cell hoping they all work. Small steps localize errors and make reasoning verifiable.
- **Never answer from memory alone.** Even when you "know" the answer (a pKa, a property, a SMILES, a benchmark number), you must verify it by executing code against an authoritative source or computation in this session. Stating a value without having produced it in the loop is a failure mode, not a shortcut.
- **Never fabricate results.** If a computation fails or returns nothing useful, say so. Do not invent numbers, citations, molecules, or structures to fill the gap.

## How far to push
Push hard. The default is to keep going.

- **Don't stop on uncertainty about approach** — that's what the loop is for. Try the most plausible method; if it doesn't work, the error tells you what to do next.
- **Don't stop because a skill is missing** — fall back to first-principles code and proceed.
- **Don't ask the user clarifying questions when a reasonable interpretation exists.** Make the assumption, state it in your final answer, and proceed. Only ask when the question is genuinely ambiguous in a way that changes the answer materially.

Stop only on real blockers:
- Required data is missing or inaccessible and cannot be derived.
- The question itself is contradictory or under-specified in a way no reasonable assumption fixes.
- A hard environment failure (library unavailable, API down) that you've already tried to work around.

When you do stop short, say exactly what's blocking and what you'd need to continue.

## Stop when you have the answer
Once the evidence supports a clear answer, stop executing. Extra runs add noise and risk introducing errors.

## Final response to the user
Reply with both the reasoning chain and the critical evidence — that's what makes an answer defensible.

Include:
- **The direct answer** to what was asked, up front.
- **The reasoning** in brief prose: what approach you took, what the key intermediate findings were, and why they support the conclusion.
- **The critical evidence**: the specific numbers, structures, plots, or table excerpts that the answer rests on. Show the load-bearing values inline, not a transcript of every cell.
- **Caveats**: assumptions you made, limitations of the method, whether any part was out-of-catalog, anything the user should know before acting on the result."""

In [ ]:
from langchain.agents import create_agent

# Define the CodeAct agent: a prebuilt ReAct loop whose only tools are read_skill and python_executor
codeact_agent = create_agent(
    model=llm,
    tools=CODEACT_TOOLS,
    system_prompt=codeact_system_prompt,
    checkpointer=InMemorySaver(),   # memory, as in Part 7, so you can ask follow-up questions
)

# specify the config (to enable short-term memory)
codeact_config = {
    'configurable': {
        'thread_id': 'codeact_conversation',
    },
    'recursion_limit': 50
}

Start the chat loop below and ask your own question, or try one of these:

- "Make a table with the Lipinski and Veber drug-likeness of Erlotinib, Gefitinib, Lapatinib, Afatinib, Osimertinib and Imatinib, and plot their pairwise structural similarity as a heatmap."
- "Align the sequences of GLP-1(7-37) (HAEGTFTSDVSSYLEGQAAKEFIAWLVKGRG) and exenatide (HGEGTFTSDLSKQMEEEAVRLFIEWLKNGGPSSGAPPPS). How identical are they, and which differences does the literature link to exenatide's resistance to DPP-4?"
- "Imatinib is given at 400 mg once daily (CL/F = 14.3 L/h, V/F = 347 L, ka = 0.6 1/h). Simulate two weeks with a missed dose on day 8. How low does the trough fall, and how many more daily doses does it take until the trough (the concentration just before a dose) is back above 90% of the steady-state trough? Plot the standard and missed-dose profiles."

In [ ]:
print('Welcome to the CodeAct agent with memory! Ask me a question.')
print('I write and run Python code, guided by skills for chemistry, biology, PK/PD, literature search and plotting.')
print('====================================================================')

while True:
    try:
        # Get input prompt from user
        user_input = input('User: ')
    except EOFError:
        break
    
    # If input contains one of three keywords: quit, exit, and q; exit the streaming process.
    if not user_input or user_input.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye!')
        break

    # Initialize graph state with user's input
    state = {
        'messages': [
            {'role': 'user', 'content': user_input}
        ]
    }

    # Print the user's message first
    print(f"""================================== User's Message ==================================
{user_input}
    """)
    
    # Supply config for the agent stream process
    for event in codeact_agent.stream(state, config=codeact_config):
        for value in event.values():
            # The last message in the state is the AI response
            msg = value["messages"][-1]
            # Use the built-in pretty_print method for better formatting
            msg.pretty_print()

### Exercise 8.3: Multi-agent system with a supervisor

One agent with many tools can get confused when the number of tools keep increasing. A cleaner pattern is to distribute the tools across different agents based on its domain. One common architecture for multi-agent is **supervisor**. In this section, we will build three sub-agents and one supervisor by hand so you can see exactly how the routing works.

| Sub-agent | Tools |
|---|---|
| **Chemist** | `resolve_smiles`, `get_properties`, `compute_similarity`, `lit_search` |
| **Biologist** | `get_drug_targets`, `get_protein_info`, `get_protein_interactions`, `analyze_protein_sequence`, `lit_search` |
| **PK/PD expert** | `simulate_pk`, `simulate_pkpd`, `lit_search` |



<img src="images/supervisor.png" width="800">

**Note on context:** a sub-agent starts every call with an empty conversation: it only sees the query the supervisor writes for it, not the user's question or the other specialists' answers. The supervisor therefore has to pass on everything a specialist needs (compound names, PK parameters, targets). Deciding what each agent sees is called *context engineering*.

**IMPORTANT:** This is a minimal toy example meant to illustrate how to build a multi-agent system. Making it production-ready would require substantial work, such as building proper tools, tuning system prompts, and doing careful context engineering both within and across agents.

In [ ]:
# Build the three specialist sub-agents
from langchain.agents import create_agent

# Reuse the `llm` created in Part 4
llm_specialist = llm

# Every specialist can search the literature
SHARED_TOOLS = [lit_search]

chemist_agent = create_agent(
    model=llm_specialist,
    tools=[resolve_smiles, get_properties, compute_similarity] + SHARED_TOOLS,
    system_prompt=(
        "You are a medicinal chemist. Resolve compound identifiers to SMILES with `resolve_smiles`, "
        "compute physicochemical properties with `get_properties` and structural similarity with "
        "`compute_similarity`. Use `lit_search` for published evidence and cite PMIDs. "
        "Reply concisely with the numbers."
    ),
)

biologist_agent = create_agent(
    model=llm_specialist,
    tools=[get_drug_targets, get_protein_info, get_protein_interactions, analyze_protein_sequence] + SHARED_TOOLS,
    system_prompt=(
        "You are a molecular biologist and pharmacologist. Look up a drug's targets, mechanism and "
        "indications with `get_drug_targets`, protein function with `get_protein_info`, interaction "
        "partners with `get_protein_interactions`, and analyse sequences with `analyze_protein_sequence`. "
        "Use `lit_search` for published evidence and cite PMIDs. Reply concisely."
    ),
)

pkpd_agent = create_agent(
    model=llm_specialist,
    tools=[simulate_pk, simulate_pkpd] + SHARED_TOOLS,
    system_prompt=(
        "You are a pharmacometrician (PK/PD expert). Simulate dosing regimens with `simulate_pk` and "
        "`simulate_pkpd`. Only use PK/PD parameters given in the request or found with `lit_search` "
        "(cite the PMID); never invent them. State the parameters and units you used, and reply "
        "concisely with the key numbers (half-life, Cmax, trough, target coverage)."
    ),
)

In [ ]:
# Wrap each sub-agent as a tool the supervisor can call
from langchain.tools import tool

def _ask(agent, query: str) -> str:
    result = agent.invoke({'messages': query})
    return result['messages'][-1].content

@tool
def ask_chemist(query: str) -> str:
    """Delegate to the chemist (SMILES, physicochemical properties, drug-likeness, structural similarity)."""
    print(f"  -> routing to Chemist: {query}")
    return _ask(chemist_agent, query)

@tool
def ask_biologist(query: str) -> str:
    """Delegate to the biologist (drug targets, mechanism of action, protein function, interactions and sequences)."""
    print(f"  -> routing to Biologist: {query}")
    return _ask(biologist_agent, query)

@tool
def ask_pkpd_expert(query: str) -> str:
    """Delegate to the PK/PD expert (simulating dosing regimens: concentrations, half-life, trough, target coverage). Include all known PK/PD parameters in the query."""
    print(f"  -> routing to PK/PD expert: {query}")
    return _ask(pkpd_agent, query)


In [ ]:
# Build the supervisor
supervisor_prompt = (
    "You are a drug-discovery research supervisor coordinating three specialists:\n"
    "  - ask_chemist     : SMILES, physicochemical properties, drug-likeness, structural similarity\n"
    "  - ask_biologist   : drug targets, mechanism of action, protein function, interactions and sequences\n"
    "  - ask_pkpd_expert : PK/PD simulations of dosing regimens\n"
    "Every specialist can also search the literature.\n"
    "For each user question, decide which specialists are needed, call them, then combine their answers "
    "into one short final reply for the user. The specialists cannot see the conversation: give each one "
    "all the information it needs (compound names, doses, PK/PD parameters with units). "
    "Do not answer scientific questions yourself — always delegate."
)

supervisor = create_agent(
    model=llm,
    tools=[ask_chemist, ask_biologist, ask_pkpd_expert],
    system_prompt=supervisor_prompt,
)

Replace the placeholder question below with one that needs more than one specialist, for example:

- "Is Erlotinib orally drug-like, what is its molecular target, and what are its steady-state Cmax and trough at 150 mg once daily (CL/F = 3.95 L/h, V/F = 233 L, ka = 0.95 1/h)?"
- "Imatinib is given at 400 mg once daily (CL/F = 14.3 L/h, V/F = 347 L, ka = 0.6 1/h). Which proteins does it inhibit, and does its steady-state trough stay above the 1 mg/L efficacy threshold?"

In [ ]:
# Try the supervisor on a question that needs more than one specialist
question = (
    "ask-the-agent-to-do-something-here"
)

print(f"USER: {question}\n")
for step, event in enumerate(supervisor.stream({'messages': question}, stream_mode='values'), start=1):
    last = event['messages'][-1]
    if getattr(last, 'tool_calls', None):
        for tc in last.tool_calls:
            print(f"[step {step}] supervisor calling {tc['name']}")
    else:
        preview = (last.content or '').strip().replace('\n', ' ')
        if len(preview) > 250:  # Modify here if you want to see full messages or longer messages
            preview = preview[:250] + '...'
        print(f"[step {step}] {type(last).__name__}: {preview}")


#### Look inside the sub-agents

The output above only shows the supervisor's steps, which hides all the each sub‑agents' message. The example code below show you how to have a look on what actually happed within each sub-agent

In [ ]:
for namespace, event in supervisor.stream({'messages': question}, stream_mode='updates', subgraphs=True):
    depth = len(namespace)
    who = 'supervisor' if depth == 0 else 'sub-agent'
    for node, update in event.items():
        if not update or 'messages' not in update:
            continue
        msg = update['messages'][-1]
        if getattr(msg, 'tool_calls', None):
            summary = 'calls ' + ', '.join(tc['name'] for tc in msg.tool_calls)
        else:
            summary = str(msg.content).strip().replace('\n', ' ')
        print(f"{'    ' * depth}[{who}:{node}] {summary}")

### Exercise 8.5: Structured output

So far the final answer has been free text. If you want to put results into a table or database, you need a **fixed schema**. Pass a Pydantic model as `response_format`: the agent still uses its tools, but its final answer is validated against the schema and returned under `result['structured_response']`.

<img src="images/react_structured_output.png" width="250">

Conceptually there is an extra *respond* step after the tool loop. `create_agent` implements it either with the provider's native structured‑output mode or, for models without one, by giving the LLM an extra "answer" tool whose arguments are your schema.

In [ ]:
from pydantic import BaseModel, Field

class DrugLikenessReport(BaseModel):
    """Lipinski Rule-of-Five assessment of one compound."""
    compound: str = Field(description="Compound name as given by the user")
    smiles: str = Field(description="SMILES used for the calculation")
    molecular_weight: float
    logp: float
    h_bond_donors: int
    h_bond_acceptors: int
    passes_lipinski: bool = Field(description="True if the compound has at most one Rule-of-Five violation")
    comment: str = Field(description="One-sentence justification")

structured_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are an expert drug discovery researcher. Use your tools to compute properties; never invent data.",
    response_format=DrugLikenessReport,
)

result = structured_agent.invoke({'messages': [{'role': 'user', 'content': 'Is Erlotinib orally drug-like?'}]})
report = result['structured_response']
print(type(report).__name__)
report

In [ ]:
# Structured responses drop straight into pandas
import pandas as pd

reports = [
    structured_agent.invoke({'messages': [{'role': 'user', 'content': f'Is {name} orally drug-like?'}]})['structured_response']
    for name in ['Imatinib', 'Dasatinib', 'Betulinic acid']
]
pd.DataFrame([r.model_dump() for r in reports])

### Exercise 8.6: Middleware

**Middleware** is the main extension point of `create_agent`. A middleware hooks into the agent loop, before/after each model call or around each tool call, without you having to rewire the graph:

<img src="images/middleware.png" width="400">


LangChain ships ready‑made middleware, for example:

| Middleware | What it does |
|---|---|
| `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` | guardrails: cap the number of LLM / tool calls per run or per thread |
| `SummarizationMiddleware` | context management: summarises old messages once the history gets long |
| `HumanInTheLoopMiddleware` | pauses before selected tool calls and waits for human approval (the `interrupt()` pattern from Exercise 8.2, ready‑made) |

You can also write your own with decorators such as `@before_model` or `@wrap_tool_call`.

Our `lit_search` tool returns up to 20 passages per call, so a long chat fills the context window quickly. The agent below adds guardrails, summarisation and a small custom logging middleware.

In [ ]:
from langchain.agents.middleware import (
    ModelCallLimitMiddleware,
    ToolCallLimitMiddleware,
    SummarizationMiddleware,
    wrap_tool_call,
)

@wrap_tool_call
def log_tool_calls(request, handler):
    """Custom middleware: print every tool call before it runs, and how long it took."""
    call = request.tool_call
    print(f"  [middleware] {call['name']}({call['args']})")
    start = time.time()
    result = handler(request)          # run the actual tool
    print(f"  [middleware] {call['name']} finished in {time.time() - start:.1f}s")
    return result

guarded_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt="You are an expert drug discovery researcher. Use your tools to answer; never invent data.",
    checkpointer=InMemorySaver(),
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),                     # at most 10 LLM calls per question
        ToolCallLimitMiddleware(tool_name='lit_search', run_limit=2),  # at most 2 literature searches per question
        SummarizationMiddleware(model=llm, trigger=('tokens', 4000), keep=('messages', 6)),
        log_tool_calls,
    ],
)

In [ ]:
guarded_config = {'configurable': {'thread_id': 'middleware-demo'}}

result = guarded_agent.invoke(
    {'messages': [{'role': 'user', 'content':
        'Run three separate literature searches: EGFR inhibitors, JAK inhibitors and BCR-ABL inhibitors. '
        'Then summarise each in one sentence with PMIDs.'}]},
    config=guarded_config,
)
print(result['messages'][-1].content)

In [ ]:
# The third lit_search was blocked by ToolCallLimitMiddleware: look for its error ToolMessage
for m in result['messages']:
    if m.type == 'tool' and m.name == 'lit_search':
        print(m.content[:150].replace('\n', ' '), '...\n')

# SummarizationMiddleware: once the history exceeds the token trigger, older messages are replaced
# by a single summary message. Ask a follow-up or two in this thread if it has not triggered yet.
history = guarded_agent.get_state(guarded_config).values['messages']
print(f"{len(history)} messages kept in the thread. The first one starts with:")
print(str(history[0].content)[:300])

#### Human‑in‑the‑loop middleware: approve the CodeAct agent's code

`HumanInTheLoopMiddleware` packages the same pattern for **any** tool: list the tools that need approval, and the agent pauses before each of their calls. Below we rebuild the CodeAct agent from Exercise 8.1 so that every `python_executor` call needs your approval, while `read_skill` (which only reads Markdown) runs freely. As in 8.2, pausing needs a checkpointer and a `thread_id`.

In [144]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

# The CodeAct agent from Exercise 8.1, now with an approval step
codeact_agent_hitl = create_agent(
    model=llm,
    tools=CODEACT_TOOLS,
    system_prompt=codeact_system_prompt,
    middleware=[
        # Pause before every python_executor call; read_skill only reads Markdown, so it runs freely
        HumanInTheLoopMiddleware(interrupt_on={'python_executor': True, 'read_skill': False}),
    ],
    checkpointer=InMemorySaver(),   # required: the paused state must be saved somewhere
)

In [ ]:
from langgraph.types import Command

def run_with_approval(agent, question: str, thread_id: str):
    """Stream an agent run; whenever it pauses for approval, show the proposed call and ask the user."""
    config = {'configurable': {'thread_id': thread_id}, 'recursion_limit': 50}
    inputs = {'messages': question}
    printed = set()   # after a resume, the paused tool call is emitted again: print each message once
    while True:
        for event in agent.stream(inputs, config=config):
            for node, update in event.items():
                if node != '__interrupt__' and update and 'messages' in update:
                    msg = update['messages'][-1]
                    if msg.id not in printed:
                        printed.add(msg.id)
                        msg.pretty_print()

        # Did the run finish, or is it paused waiting for a decision?
        state = agent.get_state(config)
        if not state.interrupts:
            break

        decisions = []
        for request in state.interrupts[0].value['action_requests']:
            print(f"\n⏸  The agent wants to run `{request['name']}`:\n")
            print(request['args'].get('code', request['args']))
            answer = input("Approve? [y]es / [n]o: ").strip().lower()
            if answer.startswith('y'):
                decisions.append({'type': 'approve'})
            else:
                decisions.append({'type': 'reject',
                                  'message': 'The user rejected this code. Explain what you intended and propose an alternative.'})
        inputs = Command(resume={'decisions': decisions})   # resume the paused run

In [ ]:
# Run it: you will be asked to approve each code execution (try rejecting one as well)
question = "request-the-agent-to-do-something-here"

print(f"USER: {question}\n")
run_with_approval(codeact_agent_hitl, question, thread_id='codeact-hitl-1')

### Exercise 8.7: Long‑term memory with a Store

The checkpointer gives **short‑term** memory: it is scoped to one `thread_id`. For knowledge that should survive across conversations, LangGraph provides a **Store**: a key‑value database organised in namespaces.

Tools reach the store through a special `runtime: ToolRuntime` parameter. LangChain injects it automatically and **hides it from the LLM's schema**, the clean way to give tools access to state, config or storage without exposing them as arguments (see the *Signature* point in Part 2).

In [ ]:
import uuid
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

store = InMemoryStore()

@tool(parse_docstring=True)
def save_finding(finding: str, runtime: ToolRuntime) -> str:
    """Save a research finding to long-term memory so it is available in future conversations.

    Args:
        finding: A short, self-contained statement worth remembering, including the numbers.
    """
    runtime.store.put(("findings",), str(uuid.uuid4()), {"text": finding})
    return "Saved."

@tool
def recall_findings(runtime: ToolRuntime) -> str:
    """Return all findings saved in previous conversations."""
    items = runtime.store.search(("findings",))
    return "\n".join(f"- {item.value['text']}" for item in items) or "No saved findings yet."

memory_agent = create_agent(
    model=llm,
    tools=[resolve_smiles, get_properties, save_finding, recall_findings],
    system_prompt=(
        "You are an expert drug discovery researcher. Use your tools; never invent data. "
        "Save important results with save_finding. When the user refers to earlier work, check recall_findings first."
    ),
    checkpointer=InMemorySaver(),
    store=store,
)

# The `runtime` parameter does not appear in the schema sent to the LLM
print(json.dumps(save_finding.tool_call_schema.model_json_schema(), indent=2))

In [ ]:
# Conversation 1: compute something and save it
r1 = memory_agent.invoke({'messages': [{'role': 'user', 'content': 'Compute the LogP of Erlotinib and save it as a finding.'}]},
                         config={'configurable': {'thread_id': 'monday'}})
print("monday :", r1['messages'][-1].content, "\n")

# Conversation 2: a NEW thread (no short-term memory of the first one), but the store is shared
r2 = memory_agent.invoke({'messages': [{'role': 'user', 'content': 'What do we already know about Erlotinib from earlier sessions?'}]},
                         config={'configurable': {'thread_id': 'tuesday'}})
print("tuesday:", r2['messages'][-1].content, "\n")

# The store can also be read directly
for item in store.search(("findings",)):
    print("store  :", item.value)

---

## Key Takeaways

- **Nodes** are functions that operate on state.
- **Edges** define the flow between nodes; **conditional edges** use a router function (or the built‑in `tools_condition`).
- **State** carries data through the agent. Use reducer functions such as `add_messages` to control how the state is updated. Messages are typed objects: `HumanMessage`, `AIMessage` (with `tool_calls`), `ToolMessage` and `SystemMessage`.
- **Tools** extend agent capabilities and are annotated with `@tool`. The LLM sees only their name, description and argument schema, so design the signature and docstring for the LLM.
- **Memory**: a checkpointer gives short‑term memory per `thread_id` (inspect it with `get_state`); a Store gives long‑term memory across threads.
- **Prebuilt agents** (`create_agent`) provide quick solutions for common patterns but offer less control than a custom graph. Extend them with `response_format` (structured output) and **middleware** (limits, summarisation, human‑in‑the‑loop, custom hooks).
- **Human‑in‑the‑loop**: `interrupt()` plus `Command(resume=...)` pauses an agent and resumes it with a human decision.
- **CodeAct** agents act by writing and executing code, guided by skills, instead of choosing from a fixed tool list.
- **Multi‑agent** systems split work across specialists, e.g. a supervisor that calls sub‑agents as tools.

---
## Resources

- **LangChain Documentation** – https://docs.langchain.com/oss/python/langchain/overview
- **LangGraph Documentation** – https://docs.langchain.com/oss/python/langgraph/overview
- **LangChain agents (`create_agent`)** – https://docs.langchain.com/oss/python/langchain/agents
- **Middleware** – https://docs.langchain.com/oss/python/langchain/middleware
- **Structured output** – https://docs.langchain.com/oss/python/langchain/structured-output
- **Streaming** – https://docs.langchain.com/oss/python/langgraph/streaming
- **Persistence (checkpointers & stores)** – https://docs.langchain.com/oss/python/langgraph/persistence
- **Interrupts / human‑in‑the‑loop** – https://docs.langchain.com/oss/python/langgraph/interrupts
- **Multi‑agent systems** – https://docs.langchain.com/oss/python/langchain/multi-agent
- **OpenAI API** – https://platform.openai.com/docs/
- **ReAct agent** - https://arxiv.org/abs/2210.03629
- **CodeAct agent** - https://arxiv.org/abs/2402.01030

These references can help you explore LangGraph and related libraries beyond the scope of this workshop.

---

**🎉 Congratulations! You've built your first AI agent with LangGraph!**

Feel free to modify and extend your agent. You can experiment with new tools, different LLMs, and additional nodes to create even more capable and personalised agents.